### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-14B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=8192,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    # token="YOUR_HF_TOKEN",  # only needed for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import textwrap
import re
import io
import contextlib
import time
import threading
import html as _html
from transformers import TextIteratorStreamer

def create_chat_interface():
    output = widgets.Output(
        layout=widgets.Layout(
            border='1px solid #ccc',
            padding='10px',
            height='600px',
            overflow_y='auto'
        )
    )

    text_input = widgets.Textarea(
        placeholder='Type your message here... (click Send to submit)',
        description='You:',
        layout=widgets.Layout(width='70%', height='80px')
    )
    send_button = widgets.Button(
        description='Send',
        button_style='primary',
        layout=widgets.Layout(width='30%', height='80px')
    )

    history = []

    def wrap_text(text, width=120):
        return textwrap.fill(text, width=width, replace_whitespace=False)

    def display_response(response):
        think_pattern = re.compile(r'<think>(.*?)</think>', re.DOTALL)
        think_match = think_pattern.search(response)
        if think_match:
            main_answer = think_pattern.sub('', response).strip()
        else:
            main_answer = response.strip()
        if main_answer:
            display(Markdown(main_answer))

    # Streaming version (yields text chunks)
    def chat_with_qwen_stream(message, history, max_new_tokens=4096, do_sample=True, temperature=0.7, top_p=0.8, top_k=20):
        messages = []
        for human, assistant in history:
            messages.append({"role": "user", "content": human})
            messages.append({"role": "assistant", "content": assistant})
        messages.append({"role": "user", "content": message})

        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextIteratorStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

        def _generate():
            model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                streamer=streamer,
            )

        t = threading.Thread(target=_generate, daemon=True)
        t.start()
        for piece in streamer:
            yield piece
        t.join()

    def on_send(b):
        user_msg = text_input.value.strip()
        if not user_msg:
            return
        text_input.value = ''

        timer = widgets.Label(value="⏱ 0.0s")
        live = widgets.HTML(value="")  # live streaming text (plain)
        done = threading.Event()

        with output:
            print("─" * 60)
            print("You:")
            print(wrap_text(user_msg))
            print()
            print("Assistant:")
            display(timer)
            display(live)

        start = time.perf_counter()

        def tick():
            while not done.is_set():
                elapsed = time.perf_counter() - start
                timer.value = f"⏱ {elapsed:.1f}s"
                time.sleep(0.1)

        threading.Thread(target=tick, daemon=True).start()

        bot_response = ""

        # ---- SPEEDUP: throttle widget updates (no logic change, just fewer UI refreshes) ----
        last_flush_t = time.perf_counter()
        FLUSH_INTERVAL_SEC = 0.10   # 30ms: smooth enough, much faster than per-token updates
        MIN_CHARS_DELTA = 64        # also flush if enough new chars accumulated
        since_last_flush_chars = 0

        def flush_live():
            # One DOM update is expensive; keep it centralized
            live.value = (
                "<pre style='white-space: pre-wrap; margin: 0;'>"
                f"{_html.escape(bot_response)}"
                "</pre>"
            )

        try:
            _buf_out = io.StringIO()
            _buf_err = io.StringIO()
            with contextlib.redirect_stdout(_buf_out), contextlib.redirect_stderr(_buf_err):
                for chunk in chat_with_qwen_stream(user_msg, history):
                    bot_response += chunk
                    since_last_flush_chars += len(chunk)

                    now = time.perf_counter()
                    if since_last_flush_chars >= MIN_CHARS_DELTA or (now - last_flush_t) >= FLUSH_INTERVAL_SEC:
                        flush_live()
                        last_flush_t = now
                        since_last_flush_chars = 0

                # final flush to ensure last bits appear
                flush_live()

        except Exception as e:
            bot_response = f"[Model Error: {e}]"
            flush_live()

        finally:
            done.set()
            elapsed = time.perf_counter() - start
            timer.value = f"⏱ {elapsed:.2f}s"

            with output:
                # Replace the live plain-text stream with final Markdown (LaTeX will render)
                live.value = ""
                display_response(bot_response)
                print(f"\n[Time: {elapsed:.2f}s]")
                print("─" * 60)
                print()

            history.append((user_msg, bot_response))

    send_button.on_click(on_send)
    input_row = widgets.HBox([text_input, send_button])
    ui = widgets.VBox([output, input_row])
    display(ui)

create_chat_interface()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
